In [2]:
import arcpy
from arcpy import env
import os
import numpy as np
from arcgis import GIS
from arcgis.features import GeoAccessor
from arcgis.features import GeoSeriesAccessor
import pandas as pd

arcpy.env.overwriteOutput = True
arcpy.env.parallelProcessingFactor = "90%"

# show all columns
pd.options.display.max_columns = None

# pd.pivot_table(df, values='a', index='b', columns='c', aggfunc='sum', fill_value=0)
# pd.DataFrame.spatial.from_featureclass(???)  
# df.spatial.to_featureclass(location=???,sanitize_columns=False)  

# gsa = arcgis.features.GeoSeriesAccessor(df['SHAPE'])  
# df['AREA'] = gsa.area  # KNOW YOUR UNITS

In [3]:
## spatial join
# target_features = ?
# join_features = ?
# output_features = os.path.join(gdb, ?)

# fieldmappings = arcpy.FieldMappings()
# fieldmappings.addTable(target_features)
# fieldmappings.addTable(join_features)

# # variable
# fieldindex = fieldmappings.findFieldMapIndex(?)
# fieldmap = fieldmappings.getFieldMap(fieldindex)
# fieldmap.mergeRule = 'Sum'
# fieldmappings.replaceFieldMap(fieldindex, fieldmap)

# sj = arcpy.SpatialJoin_analysis(target_features, join_features, output_features,'JOIN_ONE_TO_ONE', "KEEP_ALL", fieldmappings, match_option="INTERSECT")
# sj_df = pd.DataFrame.spatial.from_featureclass(sj[0]).copy()

In [4]:
# fill NA values in Spatially enabled dataframes (ignores SHAPE column)
def fill_na_sedf(df_with_shape_column, fill_value=0):
    if 'SHAPE' in list(df_with_shape_column.columns):
        cols_to_fill = df_with_shape_column.columns.difference(['SHAPE'])
        df_with_shape_column[cols_to_fill] = df_with_shape_column[cols_to_fill].fillna(fill_value)
        return df_with_shape_column
    else:
        raise Exception("Dataframe does not include 'SHAPE' column")

In [5]:
outputs = ['.\\Outputs', "scratch.gdb", 'results.gdb']

if not os.path.exists(outputs[0]):
    os.makedirs(outputs[0])

gdb = os.path.join(outputs[0], outputs[1])
gdb2 = os.path.join(outputs[0], outputs[2])

if not arcpy.Exists(gdb):
    arcpy.CreateFileGDB_management(outputs[0], outputs[1])

if not arcpy.Exists(gdb2):
    arcpy.CreateFileGDB_management(outputs[0], outputs[2])

In [6]:
parcels = pd.DataFrame.spatial.from_featureclass(r'E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\remm_base_year_data.gdb\parcels')

In [7]:
parcels.columns

Index(['OBJECTID', 'Join_Count', 'TARGET_FID', 'parcel_id', 'WFRC_parcel_id',
       'county_id', 'CO_NAME', 'year_built', 'total_market_value',
       'land_value', 'building_id', 'building_type_id', 'building_type',
       'building_sqft', 'non_residential_sqft', 'residential_units',
       'job_spaces', 'stories', 'unit_price_non_residential',
       'res_price_per_sqft', 'basebldg', 'redev_friction', 'NoBuild', 'IS_OUG',
       'parcel_acres', 'Tax_Exempt', 'parent_parcel', 'volume_one_way',
       'volume_two_way', 'volume_two_way_nofwy', 'zonal_ppa', 'x', 'y', 'note',
       'parcel_sqft', 'Split', 'Split_Factor', 'MAG_parcel_id', 'max_far',
       'max_dua', 'type1', 'type2', 'type3', 'type4', 'type5', 'type6',
       'type7', 'type8', 'TAZID_900', 'distsml_id', 'distmed_id', 'distlrg_id',
       'CITY_NAME', 'stream_dist', 'streams', 'trail_dist', 'trail',
       'airport_distance', 'airport', 'fwy_exit_dist', 'fwy_exit_new',
       'bus_stop_dist_new', 'bus_stop_new', 'bus_rte

In [8]:
mask = (parcels['county_id']==49) & ((parcels['year_built'] <= 2023) | (parcels['year_built'].isna()))
parcels[mask]['residential_units'].sum()

219214

In [ ]:
# randomly pick sf housing units in utah county that are worth more than 450k and have 1 unit to assign an ADU to
utah_high_value_sf_parcels = parcels[mask & (parcels['building_type_id'] == 1) & (parcels['residential_units']==1) & (parcels['total_market_value']>= 450000)].copy()
rng = np.random.default_rng()
utah_high_value_sf_parcels['randNumCol'] = rng.random(len(utah_high_value_sf_parcels))
utah_high_value_sf_parcels['residential_units_NEW'] = utah_high_value_sf_parcels['residential_units']

cutoff = 0.017
utah_high_value_sf_parcels.loc[utah_high_value_sf_parcels['randNumCol'] < cutoff, 'residential_units_NEW'] = 2
utah_high_value_sf_parcels.loc[utah_high_value_sf_parcels['randNumCol'] < cutoff, 'ADJUSTED'] = 1

# check the total new units
before = utah_high_value_sf_parcels['residential_units'].sum()
after = utah_high_value_sf_parcels['residential_units_NEW'].sum()
change = after - before
change

1594

In [10]:
ids_to_adjust = utah_high_value_sf_parcels[utah_high_value_sf_parcels['ADJUSTED']==1]['parcel_id'].to_list()

In [14]:
# write to text file
with open('parcel_ids_adjusted_units.txt', 'w') as f:
    for id in ids_to_adjust:
        f.write(f"{id}\n")

In [11]:
# adjust the units using list of ids
parcels.loc[parcels['parcel_id'].isin(ids_to_adjust), 'residential_units'] = 2
parcels.loc[parcels['parcel_id'].isin(ids_to_adjust), 'note'] = 'added ADU units'

In [12]:
mask = (parcels['county_id']==49) & ((parcels['year_built'] <= 2023) | (parcels['year_built'].isna()))
parcels[mask]['residential_units'].sum()

220808

In [13]:
parcels.spatial.to_featureclass(location=r'E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\remm_base_year_data.gdb\parcels_NEW',sanitize_columns=False)
parcels.drop(['SHAPE', 'OBJECTID'], axis=1).to_csv(r"E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\Tables\parcels_20260130.csv",  index=False) 